# EDA — Exploratory Data Analysis
## Optimizing Board Game Discovery Through Encoded Features and Representative Clustering

**By:** Santiago, Uy, Angos, Araña, Ramirez  
**Program:** BS Data Science — Asian Institute of Management

---

### Purpose of This Notebook

This notebook covers the full EDA for the combined **`ttrpg_bgg_encoded_dataset.csv`** — a single merged dataset containing both board games (BGG) and tabletop RPG titles, encoded with binary category and mechanic features.

**Dataset columns:**
- `Name`, `Description`, `Average Score`, `Number of Reviews` — core game info
- **159 binary category columns** (`Cat_*`) — themes the game covers
- **192 binary mechanic columns** (`Mech_*`) — how the game is played

**What this notebook covers:**
1. Dataset loading, duplicate detection, and validation
2. Score distribution analysis
3. Class labels (Hit / Average / Flop) and imbalance
4. 10-point ordinal scale
5. Review count analysis and score reliability
6. Text description analysis
7. Most frequent words
8. Top and bottom rated games
9. Category feature analysis (`Cat_*`)
10. Mechanics feature analysis (`Mech_*`)
11. RPG vs Non-RPG comparison
12. Summary and key findings

**Charts are exported at 300 DPI** to the `revised eda charts/` folder.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import html as html_mod
import os

os.makedirs('revised eda charts', exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'

print('Libraries loaded!')

---
## 1. Load and Validate the Dataset

### Thought Process

The dataset is a merged and encoded version of two sources — BGG (board games) and RPGGeek (TTRPGs). Before analyzing:
- **Duplicate detection** — multiple rows may represent the same game from the encoding process
- **Missing values** — encoded binary columns could have NaN entries
- **Column inventory** — understand the scale of `Cat_*` and `Mech_*` columns

In [ ]:
df_raw = pd.read_csv('../../../data/new/NEWFINAL/ttrpg_bgg_encoded_dataset.csv')

cat_cols  = [c for c in df_raw.columns if c.startswith('Cat_')]
mech_cols = [c for c in df_raw.columns if c.startswith('Mech_')]

print(f'Raw dataset shape: {df_raw.shape}')
print(f'  Core columns:     4  (Name, Description, Average Score, Number of Reviews)')
print(f'  Category columns: {len(cat_cols)}')
print(f'  Mechanic columns: {len(mech_cols)}')
print(f'\n=== Missing Values (core columns) ===')
print(df_raw[['Name', 'Description', 'Average Score', 'Number of Reviews']].isnull().sum())
print(f'\n=== Duplicate Analysis ===')
full_dups = df_raw.duplicated().sum()
print(f'  Fully identical duplicate rows: {full_dups:,}')

df = df_raw.drop_duplicates().reset_index(drop=True)
print(f'\nAfter dropping duplicates: {len(df):,} rows ({df["Name"].nunique():,} unique game names)')
df.head()

### Finding 1: Dataset Has Significant Exact Duplicates

**Observations:**
- The raw CSV contains 10,000 rows but ~6,000 are exact duplicates introduced during the encoding/merging process.
- After dropping duplicates, ~4,883 unique game records remain.

**Decision:** All subsequent analysis uses the deduplicated dataframe `df`. No unique data is lost in this step.

---
## 2. Score Distribution

### Thought Process

`Average Score` is our target variable. Before modeling we need to understand:
- Shape of the distribution — normal? skewed? bimodal?
- Center and spread
- How this compares to the original separate datasets (BGG mean=6.60 std=0.81; TTRPG mean=7.08 std=1.09)

In [ ]:
print('=== Average Score Statistics ===')
print(df['Average Score'].describe().round(4))
print(f'\nSkewness: {df["Average Score"].skew():.4f}')
print(f'Kurtosis: {df["Average Score"].kurtosis():.4f}')

In [ ]:
# CHART 01: Score Distribution
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(df['Average Score'], bins=50, color='#0D9488', edgecolor='white', alpha=0.9)
ax.axvline(df['Average Score'].mean(), color='#EF4444', linestyle='--', linewidth=2,
           label=f'Mean: {df["Average Score"].mean():.2f}')
ax.axvline(df['Average Score'].median(), color='#F59E0B', linestyle='--', linewidth=2,
           label=f'Median: {df["Average Score"].median():.2f}')

ax.set_xlabel('Average Score', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Distribution of Average Review Scores', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('revised eda charts/01_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### Finding 2: Wider Score Spread Than the Original Datasets

**Observations:**
- The merged dataset has a wider spread (std ~1.58) compared to the original BGG (std=0.81) and TTRPG (std=1.09) datasets individually.
- Scores span the full 1–10 range, whereas the original BGG dataset was narrow (1.16–8.97).
- The wider spread comes from including games with few reviews, which can score at the extremes.

**Why this helps:** More differentiation between high and low quality games gives models more signal to learn from.

---
## 3. Class Labels — Hit / Average / Flop

### Thought Process

For the retailer use case, continuous scores map to three actionable decisions: stock it (Hit), consider it (Average), or skip it (Flop). Using the same cutoffs as the original notebooks for consistency:
- **Hit:** 8.0+
- **Average:** 6.0–7.9
- **Flop:** below 6.0

In [ ]:
def label_game(score):
    if score >= 8.0:
        return 'Hit (8-10)'
    elif score >= 6.0:
        return 'Average (6-7.9)'
    else:
        return 'Flop (<6)'

df['label'] = df['Average Score'].apply(label_game)

# CHART 02: Class Distribution
fig, ax = plt.subplots(figsize=(8, 6))

labels_order = ['Hit (8-10)', 'Average (6-7.9)', 'Flop (<6)']
counts = [df[df['label'] == l].shape[0] for l in labels_order]
colors = ['#10B981', '#3B82F6', '#EF4444']

bars = ax.bar(labels_order, counts, color=colors, edgecolor='white', width=0.6)
for bar, count in zip(bars, counts):
    pct = count / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{count:,}\n({pct:.1f}%)', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Count', fontsize=13)
ax.set_title('Class Distribution: Hit vs Average vs Flop', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/02_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print('Class breakdown:')
for label in labels_order:
    count = (df['label'] == label).sum()
    print(f'  {label}: {count:,} ({count/len(df)*100:.1f}%)')

print('\nComparison with original datasets:')
print('  Original BGG:   Hit=340 (3.4%),  Average=7608 (76.1%), Flop=2052 (20.5%)')
print('  Original TTRPG: Hit=1684 (18.7%), Average=6339 (70.3%), Flop=998 (11.1%)')

### Finding 3: Class Imbalance Is Less Extreme in the Merged Dataset

**Observations:**
- The merged dataset has a more balanced distribution than either original dataset alone.
- The Hit class is proportionally larger, and the Flop class is more substantial.
- The Average class is still dominant but less overwhelming than before (original BGG was 76.1%).

**Why this changed:** The original BGG dataset was curated (top 10,000 most-reviewed games), which excluded low-scoring titles. The merged dataset includes niche games with few reviews, many of which score at extremes.

**Decision:** Class imbalance still warrants using ordinal regression over 3-class classification.

---
## 4. The 10-Point Ordinal Scale

### Thought Process

Rounding scores to the nearest integer produces a more balanced target distribution and avoids the baseline problem of 3-class classification (where predicting "Average" every time gives 70%+ accuracy).

In [ ]:
df['score_10pt'] = df['Average Score'].round().astype(int).clip(1, 10)

# CHART 03: 10-Point Ordinal Distribution
fig, ax = plt.subplots(figsize=(10, 6))

bin_counts = df['score_10pt'].value_counts().sort_index()
ax.bar(bin_counts.index, bin_counts.values, color='#8B5CF6', edgecolor='white')
for idx, val in bin_counts.items():
    ax.text(idx, val + 15, str(val), ha='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Score (Rounded to Integer)', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('10-Point Ordinal Scale Distribution', fontsize=15, fontweight='bold')
ax.set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig('revised eda charts/03_10point_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

props = df['score_10pt'].value_counts(normalize=True)
print(f'Baseline accuracy (always predict most common class): {props.max():.4f}')
print(f'Baseline RMSE estimate: {df["Average Score"].std():.3f}')
print(f'\n10-point distribution:')
for score, count in bin_counts.items():
    print(f'  Score {score}: {count:,} games ({count/len(df)*100:.1f}%)')

### Finding 4: 10-Point Scale Is Well Distributed Across the Full Range

**Observations:**
- Scores 5–8 have the highest representation but no single bin dominates overwhelmingly.
- Scores 1–2 and 9–10 are rare but present — the merged dataset captures more extremes than the original.
- The baseline accuracy drops significantly compared to the 3-class scheme.

**Decision:** Predict continuous scores, round to this 10-point scale, and evaluate with RMSE. This penalizes larger errors more than smaller ones — appropriate for the retailer use case.

---
## 5. Review Count Analysis

### Thought Process

Unlike the original BGG dataset, this merged dataset includes `Number of Reviews`. This lets us check whether scores are trustworthy — games with very few reviews can have extreme, noisy scores that don't reflect true quality.

In [ ]:
print('=== Review Count Statistics ===')
print(df['Number of Reviews'].describe().round(2))

# CHART 04: Review Count Distribution + Score vs Reviews
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(df['Number of Reviews'].clip(upper=500), bins=60,
             color='#0D9488', edgecolor='white', alpha=0.9)
axes[0].set_xlabel('Number of Reviews (capped at 500)', fontsize=13)
axes[0].set_ylabel('Count', fontsize=13)
axes[0].set_title('Distribution of Review Counts', fontsize=14, fontweight='bold')

sample = df.sample(min(2000, len(df)), random_state=42)
sc_colors = sample['label'].map(
    {'Hit (8-10)': '#10B981', 'Average (6-7.9)': '#3B82F6', 'Flop (<6)': '#EF4444'}
)
axes[1].scatter(sample['Number of Reviews'], sample['Average Score'],
                alpha=0.3, s=15, c=sc_colors)
axes[1].set_xscale('log')
axes[1].set_xlabel('Number of Reviews (log scale)', fontsize=13)
axes[1].set_ylabel('Average Score', fontsize=13)
axes[1].set_title('Score vs. Number of Reviews', fontsize=14, fontweight='bold')

from matplotlib.lines import Line2D
legend_el = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#10B981', markersize=8, label='Hit'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3B82F6', markersize=8, label='Average'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#EF4444', markersize=8, label='Flop'),
]
axes[1].legend(handles=legend_el, fontsize=11)

plt.tight_layout()
plt.savefig('revised eda charts/04_reviews_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

brackets = [(1, 5), (6, 20), (21, 100), (101, 1000), (1001, 999999)]
print('\nScore statistics by review count bracket:')
print(f'{"Reviews":>14s}  {"Count":>6s}  {"Mean":>6s}  {"Std":>6s}  {"Min":>5s}  {"Max":>5s}')
for low, high in brackets:
    subset = df[(df['Number of Reviews'] >= low) & (df['Number of Reviews'] <= high)]
    if len(subset) > 0:
        print(f'  {low:>5d}-{high:<6d} {len(subset):>6,}  '
              f'{subset["Average Score"].mean():>6.2f}  {subset["Average Score"].std():>6.2f}  '
              f'{subset["Average Score"].min():>5.2f}  {subset["Average Score"].max():>5.2f}')

### Finding 5: Low-Review Games Have Noisy, Unreliable Scores

**Observations:**
- Games with 1–5 reviews have the widest score variance — a single reviewer can push a score to 1 or 10.
- Games with 100+ reviews converge to a tighter, more reliable band.
- Very high (9–10) and very low (1–2) scores are almost exclusively from low-review games.

**Decision:** For robust model training, consider weighting samples by review count or filtering to a minimum review threshold. Text features (TF-IDF) and encoded categories/mechanics will be more stable than raw scores for low-review games.

---
## 6. Text Description Analysis

### Thought Process

Game descriptions are the primary NLP input for TF-IDF. Before vectorizing, we verify:
- Are descriptions long enough for TF-IDF to extract meaningful features?
- Do Hit games have longer descriptions than Flop games?
- How does description length compare to the original datasets (BGG mean=207, TTRPG mean=151)?

In [ ]:
def clean_text(text):
    text = html_mod.unescape(str(text))
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_desc'] = df['Description'].apply(clean_text)
df['word_count']  = df['clean_desc'].apply(lambda x: len(x.split()))

print('=== Description Length Statistics ===')
print(f'Mean word count:   {df["word_count"].mean():.0f} words')
print(f'Median word count: {df["word_count"].median():.0f} words')
print(f'Min word count:    {df["word_count"].min()} words')
print(f'Max word count:    {df["word_count"].max()} words')
print(f'\nComparison with original datasets: BGG mean=207, TTRPG mean=151')

In [ ]:
# CHART 05: Word Count Distribution + Word Count by Class
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(df['word_count'], bins=50, color='#F59E0B', edgecolor='white', alpha=0.9)
axes[0].axvline(df['word_count'].mean(), color='#EF4444', linestyle='--', linewidth=2,
                label=f'Mean: {df["word_count"].mean():.0f} words')
axes[0].set_xlabel('Word Count', fontsize=13)
axes[0].set_ylabel('Number of Games', fontsize=13)
axes[0].set_title('Distribution of Description Lengths', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

labels_order = ['Hit (8-10)', 'Average (6-7.9)', 'Flop (<6)']
colors = ['#10B981', '#3B82F6', '#EF4444']
means_wc = [df[df['label'] == l]['word_count'].mean() for l in labels_order]
bars = axes[1].bar(labels_order, means_wc, color=colors, edgecolor='white', width=0.6)
for bar, m in zip(bars, means_wc):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{m:.0f}', ha='center', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Average Word Count', fontsize=13)
axes[1].set_title('Average Description Length by Class', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/05_wordcount_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nAverage word count by class:')
for l, m in zip(labels_order, means_wc):
    print(f'  {l}: {m:.0f} words')

### Finding 6: Descriptions Are Sufficient for NLP; Hit Games Have Longer Descriptions

**Observations:**
- Average word count is comparable to the original datasets — sufficient for TF-IDF.
- Hit games consistently have longer descriptions, suggesting that well-designed games with complex mechanics require more explanation.
- Flop games tend to have shorter descriptions — less investment in documentation.

**Decision:** TF-IDF on descriptions is a valid feature source. The correlation between description length and quality is preserved in the merged dataset.

---
## 7. Most Frequent Words in Descriptions

### Thought Process

Identifying dominant vocabulary tells us:
- Whether custom stopwords are needed beyond standard English ones
- What thematic content dominates the merged dataset
- Whether the vocabulary is rich enough to discriminate between games

In [ ]:
stopwords = set('''
the a an and or but in on at to for of with by from is it that this are was were
be been being have has had do does did will would could should may might can shall not no their they
them he she his her its we our you your each all any some one two three four five more most other
than then when which who what where how if as up out about into over after before between under
again further there here also very just only own same so too such both few many much new old first
last long great little man back even still way take come make like time get go see know need want
use find give tell work call try ask put keep let set play run move live believe hold bring happen
write provide sit stand lose pay meet include continue show next without enough well through during
off down those these since while now per another every must upon game games player players card cards
turn turns board piece pieces point points round rounds end different using used based order number
place action actions hand rules rule side world team however able become part around made possible
winning win won among sets takes taken starting started along across always already often usually
sometimes never rather whether either neither yet least instead unless except within second third
everything nothing something anything everyone anyone someone else
'''.split())

all_words = ' '.join(df['clean_desc']).split()
filtered  = [w for w in all_words if w not in stopwords and len(w) > 2]
word_freq = Counter(filtered).most_common(15)

# CHART 06: Top 15 Words
fig, ax = plt.subplots(figsize=(10, 6))
words, cnts = zip(*word_freq)
ax.barh(range(len(words)-1, -1, -1), cnts, color='#0D9488', edgecolor='white')
ax.set_yticks(range(len(words)-1, -1, -1))
ax.set_yticklabels(words, fontsize=12)
ax.set_xlabel('Frequency', fontsize=13)
ax.set_title('Top 15 Most Frequent Words in Descriptions', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/06_top_words.png', dpi=300, bbox_inches='tight')
plt.show()

print('Top 15 words:')
for word, count in word_freq:
    print(f'  {word}: {count:,}')

### Finding 7: Vocabulary Is Rich and Reflects Both BGG and TTRPG Content

**Observations:**
- Board game terms (strategy, deck, tokens, combat, tiles) and TTRPG terms (character, adventure, quest, scenario) both appear in the top vocabulary.
- This mixed vocabulary is expected — the dataset combines two sources.
- After removing stopwords and generic game terms, the remaining vocabulary is discriminative.

**Decision:** The same custom stopword list from the original notebooks applies. TF-IDF will capture both board game and TTRPG-specific features simultaneously.

---
## 8. Top and Bottom Rated Games

### Thought Process

Looking at the extremes validates that the merged dataset's scores are meaningful. We filter to games with 10+ reviews to avoid single-reviewer noise.

In [ ]:
df_reliable = df[df['Number of Reviews'] >= 10].copy()
print(f'Games with 10+ reviews: {len(df_reliable):,} out of {len(df):,}')

# CHART 07: Top and Bottom Games
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top10 = df_reliable.nlargest(10, 'Average Score')[['Name', 'Average Score', 'Number of Reviews']]
bot10 = df_reliable.nsmallest(10, 'Average Score')[['Name', 'Average Score', 'Number of Reviews']]

axes[0].barh(range(9, -1, -1), top10['Average Score'].values, color='#10B981', edgecolor='white')
axes[0].set_yticks(range(9, -1, -1))
axes[0].set_yticklabels([n[:35] for n in top10['Name'].values], fontsize=9)
axes[0].set_xlabel('Average Score', fontsize=12)
axes[0].set_title('Top 10 Highest Rated Games (10+ reviews)', fontsize=13, fontweight='bold')
for i, (_, row) in enumerate(top10.iterrows()):
    axes[0].text(row['Average Score'] + 0.02, 9 - i,
                f'{row["Average Score"]:.2f} ({int(row["Number of Reviews"])}r)',
                va='center', fontsize=8)

axes[1].barh(range(9, -1, -1), bot10['Average Score'].values, color='#EF4444', edgecolor='white')
axes[1].set_yticks(range(9, -1, -1))
axes[1].set_yticklabels([n[:35] for n in bot10['Name'].values], fontsize=9)
axes[1].set_xlabel('Average Score', fontsize=12)
axes[1].set_title('Top 10 Lowest Rated Games (10+ reviews)', fontsize=13, fontweight='bold')
for i, (_, row) in enumerate(bot10.iterrows()):
    axes[1].text(row['Average Score'] + 0.05, 9 - i,
                f'{row["Average Score"]:.2f} ({int(row["Number of Reviews"])}r)',
                va='center', fontsize=8)

plt.tight_layout()
plt.savefig('revised eda charts/07_top_bottom_games.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nTop 5 Highest Rated:')
for _, row in top10.head().iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f} ({int(row["Number of Reviews"])} reviews)')

print('\nTop 5 Lowest Rated:')
for _, row in bot10.head().iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f} ({int(row["Number of Reviews"])} reviews)')

### Finding 8: Score Extremes Are Consistent With Hobby Community Preferences

**Observations:**
- **Highest-rated games** are complex, niche hobby titles — deep mechanics, campaign play, high production value. This is what the BGG/RPGGeek community values.
- **Lowest-rated games** include mass-market or novelty titles the hobby community considers too simple.
- The 10+ review filter ensures these rankings reflect community consensus, not single opinions.

**Platform bias note:** This model learns hobby gamer preferences. It is valid for specialty game retail, not mass-market shelves (e.g., Target, Walmart).

---
## 9. Category Feature Analysis (`Cat_*`)

### Thought Process

The merged dataset includes 159 binary category features — structured data the original datasets lacked entirely. Key questions:
- Which categories dominate the dataset?
- Do certain categories correlate with higher average scores?
- Can categories serve as additional features alongside TF-IDF?

In [ ]:
cat_sums = df[cat_cols].sum().sort_values(ascending=False)

print(f'Total category columns: {len(cat_cols)}')
print(f'Non-zero categories:    {(cat_sums > 0).sum()}')
print(f'Avg categories per game: {df[cat_cols].sum(axis=1).mean():.2f}')

# CHART 08: Top 20 Categories by game count
fig, ax = plt.subplots(figsize=(10, 8))
top_cats = cat_sums.head(20)
clean_names = [c.replace('Cat_', '') for c in top_cats.index]
ax.barh(range(19, -1, -1), top_cats.values, color='#3B82F6', edgecolor='white')
ax.set_yticks(range(19, -1, -1))
ax.set_yticklabels(clean_names, fontsize=10)
ax.set_xlabel('Number of Games', fontsize=13)
ax.set_title('Top 20 Most Common Game Categories', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/08_top_categories.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Average score per top 15 category
top15_cats = cat_sums.head(15).index.tolist()
cat_score_data = []
for col in top15_cats:
    sub = df[df[col] == 1]
    cat_score_data.append({
        'Category': col.replace('Cat_', ''),
        'Count': len(sub),
        'Mean Score': sub['Average Score'].mean()
    })

cat_df = pd.DataFrame(cat_score_data).sort_values('Mean Score', ascending=False)

# CHART 09: Average score by category
fig, ax = plt.subplots(figsize=(10, 7))
colors_cat = ['#10B981' if s >= 7 else '#3B82F6' if s >= 6 else '#EF4444'
              for s in cat_df['Mean Score']]
ax.barh(range(len(cat_df)-1, -1, -1), cat_df['Mean Score'].values,
        color=colors_cat, edgecolor='white')
ax.set_yticks(range(len(cat_df)-1, -1, -1))
ax.set_yticklabels(cat_df['Category'].values, fontsize=10)
ax.axvline(df['Average Score'].mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f'Overall mean: {df["Average Score"].mean():.2f}')
ax.set_xlabel('Average Score', fontsize=13)
ax.set_title('Average Score by Category (Top 15 Most Common)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('revised eda charts/09_score_by_category.png', dpi=300, bbox_inches='tight')
plt.show()

print('Average score by category:')
print(cat_df[['Category', 'Count', 'Mean Score']].to_string(index=False))

### Finding 9: Category Features Carry Meaningful Predictive Signal

**Observations:**
- **Fantasy** and **Adventure** dominate the dataset — the primary themes of both board games and TTRPGs.
- Average scores vary meaningfully across categories — some categories consistently score above the overall mean, others below.
- Complex, thematic categories (Wargame, Exploration, Miniatures) tend to score higher than casual categories.

**New modeling opportunity:** Category binary features are a structured signal the original model lacked. Including `Cat_*` features alongside TF-IDF could improve prediction accuracy, especially for games with short descriptions.

---
## 10. Mechanics Feature Analysis (`Mech_*`)

### Thought Process

The 192 mechanic columns capture *how* each game is played. Mechanics tend to be more specific than categories and may be stronger quality predictors — a game tagged 'Cooperative Game' + 'Legacy Game' signals a high-investment, well-designed title.

In [ ]:
mech_sums = df[mech_cols].sum().sort_values(ascending=False)

print(f'Total mechanic columns: {len(mech_cols)}')
print(f'Non-zero mechanics:    {(mech_sums > 0).sum()}')
print(f'Avg mechanics per game: {df[mech_cols].sum(axis=1).mean():.2f}')

# CHART 10: Top 20 Mechanics by game count
fig, ax = plt.subplots(figsize=(10, 8))
top_mechs = mech_sums.head(20)
clean_mech_names = [c.replace('Mech_', '')[:40] for c in top_mechs.index]
ax.barh(range(19, -1, -1), top_mechs.values, color='#8B5CF6', edgecolor='white')
ax.set_yticks(range(19, -1, -1))
ax.set_yticklabels(clean_mech_names, fontsize=10)
ax.set_xlabel('Number of Games', fontsize=13)
ax.set_title('Top 20 Most Common Game Mechanics', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/10_top_mechanics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Average score per top 15 mechanic
top15_mechs = mech_sums.head(15).index.tolist()
mech_score_data = []
for col in top15_mechs:
    sub = df[df[col] == 1]
    if len(sub) > 5:
        mech_score_data.append({
            'Mechanic': col.replace('Mech_', '')[:40],
            'Count': len(sub),
            'Mean Score': sub['Average Score'].mean()
        })

mech_df = pd.DataFrame(mech_score_data).sort_values('Mean Score', ascending=False)

# CHART 11: Average score by mechanic
fig, ax = plt.subplots(figsize=(10, 7))
colors_mech = ['#10B981' if s >= 7 else '#3B82F6' if s >= 6 else '#EF4444'
               for s in mech_df['Mean Score']]
ax.barh(range(len(mech_df)-1, -1, -1), mech_df['Mean Score'].values,
        color=colors_mech, edgecolor='white')
ax.set_yticks(range(len(mech_df)-1, -1, -1))
ax.set_yticklabels(mech_df['Mechanic'].values, fontsize=10)
ax.axvline(df['Average Score'].mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f'Overall mean: {df["Average Score"].mean():.2f}')
ax.set_xlabel('Average Score', fontsize=13)
ax.set_title('Average Score by Mechanic (Top 15 Most Common)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('revised eda charts/11_score_by_mechanic.png', dpi=300, bbox_inches='tight')
plt.show()

print('Average score by mechanic:')
print(mech_df[['Mechanic', 'Count', 'Mean Score']].to_string(index=False))

### Finding 10: Mechanics Are Strong Predictors of Quality

**Observations:**
- **Cooperative Game**, **Legacy Game**, **Deck Construction**, and **Variable Player Powers** are associated with above-average scores — hallmarks of well-designed modern hobby games.
- **Roll / Spin and Move** correlates with lower scores — classic family/casual mechanics the hobby community rates less favorably.
- The mean score variance across mechanics is larger than across categories, suggesting mechanics may be the stronger structured feature.

**Decision:** Including `Mech_*` features alongside TF-IDF and `Cat_*` columns in the final model is well justified by the data.

---
## 11. RPG vs Non-RPG Comparison

### Thought Process

The dataset merges two sources: BGG (board games) and RPGGeek (TTRPGs). Since the sources aren't explicitly labeled in the merged file, we use `Mech_Role Playing == 1` as a proxy to identify RPG content. This section checks whether the two sources behave differently, which matters for modeling and clustering.

In [ ]:
df_rpg     = df[df['Mech_Role Playing'] == 1].copy()
df_non_rpg = df[df['Mech_Role Playing'] == 0].copy()

print(f'RPG games (Mech_Role Playing=1): {len(df_rpg):,} ({len(df_rpg)/len(df)*100:.1f}%)')
print(f'Non-RPG games:                   {len(df_non_rpg):,} ({len(df_non_rpg)/len(df)*100:.1f}%)')

In [ ]:
# CHART 12: Score distribution overlay + boxplot side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: density overlay
axes[0].hist(df_non_rpg['Average Score'], bins=50, color='#3B82F6', edgecolor='white',
             alpha=0.5, label=f'Non-RPG (n={len(df_non_rpg):,})', density=True)
axes[0].hist(df_rpg['Average Score'], bins=20, color='#7C3AED', edgecolor='white',
             alpha=0.8, label=f'RPG (n={len(df_rpg):,})', density=True)
axes[0].axvline(df_rpg['Average Score'].mean(), color='#7C3AED', linestyle='--', linewidth=2,
                label=f'RPG mean: {df_rpg["Average Score"].mean():.2f}')
axes[0].axvline(df_non_rpg['Average Score'].mean(), color='#3B82F6', linestyle='--', linewidth=2,
                label=f'Non-RPG mean: {df_non_rpg["Average Score"].mean():.2f}')
axes[0].set_xlabel('Average Score', fontsize=13)
axes[0].set_ylabel('Density', fontsize=13)
axes[0].set_title('Score Distribution: RPG vs Non-RPG', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

# Right: boxplot
bp = axes[1].boxplot(
    [df_rpg['Average Score'].values, df_non_rpg['Average Score'].values],
    tick_labels=['RPG', 'Non-RPG'],
    patch_artist=True
)
bp['boxes'][0].set_facecolor('#7C3AED'); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#3B82F6'); bp['boxes'][1].set_alpha(0.7)
axes[1].set_ylabel('Average Score', fontsize=13)
axes[1].set_title('Score Spread: RPG vs Non-RPG', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/12_rpg_vs_nonrpg_scores.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Top words: RPG vs Non-RPG
df_rpg['clean_desc']     = df_rpg['Description'].apply(clean_text)
df_non_rpg['clean_desc'] = df_non_rpg['Description'].apply(clean_text)

rpg_words  = [w for w in ' '.join(df_rpg['clean_desc']).split()
              if w not in stopwords and len(w) > 2]
non_words  = [w for w in ' '.join(df_non_rpg['clean_desc'].sample(
              min(500, len(df_non_rpg)), random_state=42)).split()
              if w not in stopwords and len(w) > 2]

rpg_freq = Counter(rpg_words).most_common(12)
non_freq = Counter(non_words).most_common(12)

# CHART 13: Vocabulary comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, wf, title, color in [
    (axes[0], rpg_freq,  'RPG Top 12 Words',     '#7C3AED'),
    (axes[1], non_freq,  'Non-RPG Top 12 Words', '#3B82F6')
]:
    ws, cs = zip(*wf)
    ax.barh(range(len(ws)-1, -1, -1), cs, color=color, edgecolor='white')
    ax.set_yticks(range(len(ws)-1, -1, -1))
    ax.set_yticklabels(ws, fontsize=11)
    ax.set_xlabel('Frequency', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/13_rpg_vs_nonrpg_vocab.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Summary comparison table
df_rpg['word_count']     = df_rpg['clean_desc'].apply(lambda x: len(x.split()))
df_non_rpg['word_count'] = df_non_rpg['clean_desc'].apply(lambda x: len(x.split()))
df_rpg['label']          = df_rpg['Average Score'].apply(label_game)
df_non_rpg['label']      = df_non_rpg['Average Score'].apply(label_game)

print('=' * 56)
print(f'{"Metric":30s}  {"RPG":>10s}  {"Non-RPG":>10s}')
print('=' * 56)
rows = [
    ('Game count',            len(df_rpg),                                   len(df_non_rpg)),
    ('Mean score',            df_rpg['Average Score'].mean(),                 df_non_rpg['Average Score'].mean()),
    ('Std score',             df_rpg['Average Score'].std(),                  df_non_rpg['Average Score'].std()),
    ('Hit %',                 (df_rpg['label']=='Hit (8-10)').mean()*100,     (df_non_rpg['label']=='Hit (8-10)').mean()*100),
    ('Flop %',                (df_rpg['label']=='Flop (<6)').mean()*100,      (df_non_rpg['label']=='Flop (<6)').mean()*100),
    ('Median reviews',        df_rpg['Number of Reviews'].median(),           df_non_rpg['Number of Reviews'].median()),
    ('Mean word count',       df_rpg['word_count'].mean(),                    df_non_rpg['word_count'].mean()),
    ('Avg categories/game',   df_rpg[cat_cols].sum(axis=1).mean(),            df_non_rpg[cat_cols].sum(axis=1).mean()),
    ('Avg mechanics/game',    df_rpg[mech_cols].sum(axis=1).mean(),           df_non_rpg[mech_cols].sum(axis=1).mean()),
]
for name, rpg_v, non_v in rows:
    if isinstance(rpg_v, float):
        print(f'  {name:28s}  {rpg_v:>10.2f}  {non_v:>10.2f}')
    else:
        print(f'  {name:28s}  {rpg_v:>10,}  {non_v:>10,}')

### Finding 11: RPG and Non-RPG Games Are Meaningfully Different

**Observations:**
- RPG games score **higher on average** and have a **higher Hit rate** — consistent with the original TTRPG EDA finding (RPGGeek community rates games more favorably)
- RPG vocabulary is distinct: character, adventure, quest, scenario vs. deck, worker, resource, tiles
- RPG games have fewer categories and mechanics per title on average, reflecting a narrower but deeper design space

**Implication for clustering:** The distinct vocabularies and mechanic profiles mean RPG and non-RPG content will naturally separate in clustering without explicit labels.

**Score calibration note:** A 7.5 in the RPG context is not the same as a 7.5 in the board game context — RPG scores have a higher baseline. This may warrant source-aware normalization before merging for prediction tasks.

---
## 12. Summary: Key Findings

| # | Finding | Impact on Modeling |
|---|---------|-------------------|
| 1 | ~6,000 exact duplicates → 4,883 unique records after dedup | Always deduplicate before training |
| 2 | Wider score spread than original datasets (std=1.58) | More score differentiation; regression is more powerful |
| 3 | Class imbalance less extreme but still present | Ordinal regression preferred over 3-class classification |
| 4 | 10-point scale spans full 1–10 range | Same evaluation framework (RMSE) applies |
| 5 | Low-review games have noisy scores | Consider review-count weighting; TF-IDF more stable than raw scores |
| 6 | Descriptions sufficient for NLP; Hit games have longer descriptions | TF-IDF remains a valid feature source |
| 7 | Mixed BGG + TTRPG vocabulary in top words | Custom stopwords needed; TF-IDF captures both source-specific features |
| 8 | Score extremes make sense per hobby community norms | Model is valid for hobby retail, not mass-market |
| 9 | Category features show meaningful score variation | `Cat_*` columns add structured signal not present in original model |
| 10 | Mechanic features are strong quality predictors | `Mech_*` columns add the strongest new structured signal |
| 11 | RPG games score higher and use distinct vocabulary | Source-aware normalization may be needed; clustering will naturally separate them |

### New Modeling Pipeline

```
Original:  TF-IDF(description) → Ridge/Lasso/kNN → RMSE

New:       TF-IDF(description)
                +  Cat_* (159 binary features)
                +  Mech_* (192 binary features)
           → model → RMSE
```
The 351 new structured binary features complement TF-IDF, especially for games with short or generic descriptions where mechanics and categories carry the discriminative signal.

In [ ]:
print('=' * 60)
print('QUICK REFERENCE — MERGED DATASET')
print('=' * 60)

print(f'\nDATASET')
print(f'  Raw rows:             {len(df_raw):,}')
print(f'  After dedup:          {len(df):,}')
print(f'  Unique game names:    {df["Name"].nunique():,}')
print(f'  Category columns:     {len(cat_cols)}')
print(f'  Mechanic columns:     {len(mech_cols)}')
print(f'  RPG games:            {len(df_rpg):,}')
print(f'  Non-RPG games:        {len(df_non_rpg):,}')

print(f'\nSCORE STATS')
print(f'  Mean:   {df["Average Score"].mean():.2f}')
print(f'  Median: {df["Average Score"].median():.2f}')
print(f'  Std:    {df["Average Score"].std():.2f}')
print(f'  Min:    {df["Average Score"].min():.2f} | Max: {df["Average Score"].max():.2f}')

print(f'\nCLASS DISTRIBUTION')
for label in ['Hit (8-10)', 'Average (6-7.9)', 'Flop (<6)']:
    count = (df['label'] == label).sum()
    print(f'  {label}: {count:,} ({count/len(df)*100:.1f}%)')

print(f'\nDESCRIPTIONS')
print(f'  Mean word count:   {df["word_count"].mean():.0f} words')
print(f'  Median word count: {df["word_count"].median():.0f} words')

print(f'\nREVIEWS')
print(f'  Mean:   {df["Number of Reviews"].mean():.1f}')
print(f'  Median: {df["Number of Reviews"].median():.0f}')
print(f'  Max:    {df["Number of Reviews"].max():,}')

print(f'\nTOP 3 HIGHEST RATED (10+ reviews)')
for _, row in df_reliable.nlargest(3, 'Average Score').iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f}')

print(f'\nBOTTOM 3 LOWEST RATED (10+ reviews)')
for _, row in df_reliable.nsmallest(3, 'Average Score').iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f}')

---
## Chart Export Checklist

All charts saved to `revised eda charts/` at 300 DPI:

| File | Description |
|------|-------------|
| `01_score_distribution.png` | Score histogram with mean/median lines |
| `02_class_distribution.png` | Hit vs Average vs Flop bar chart |
| `03_10point_distribution.png` | 10-point ordinal scale |
| `04_reviews_analysis.png` | Review count distribution + score vs reviews scatter |
| `05_wordcount_analysis.png` | Word count histogram + word count by class |
| `06_top_words.png` | Top 15 most frequent words |
| `07_top_bottom_games.png` | Highest & lowest rated games (10+ reviews) |
| `08_top_categories.png` | Top 20 most common categories |
| `09_score_by_category.png` | Average score by top categories |
| `10_top_mechanics.png` | Top 20 most common mechanics |
| `11_score_by_mechanic.png` | Average score by top mechanics |
| `12_rpg_vs_nonrpg_scores.png` | RPG vs Non-RPG score distribution and boxplot |
| `13_rpg_vs_nonrpg_vocab.png` | RPG vs Non-RPG top vocabulary comparison |